# ASSIGNMENT 4: Graph Generation from an Architectural Model — Gleis21 Standard Floor

## Overview
This assignment converts the Gleis21 building standard (type) floor into a graph
representation. By analyzing the spatial relationships between rooms and doors, we build a
network structure that captures both the geometric and topological properties of the
architecture.

## Objectives
- Load and visualize the Gleis21 standard-floor model exported from Grasshopper
- Define nodes and edges based on architectural elements (rooms and door apertures)
- Generate a graph representation of spatial relationships
- Visualize the graph structure with nodes and edges
- Document the mapping logic between architectural elements and graph components

## 1. Import Libraries
Import TopologicPy modules for geometric and graph operations.

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\giofo\Desktop\ARCHIVIO\MASTERS\MACAD\2_MACAD\0_MACAD DRIVE\AIA\Graph_ML-Giovanni_Carlo_Volpe\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Verify TopologicPy Version
Check the installed version of TopologicPy.

In [2]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.34) is OLDER than the latest version (0.9.51) from PyPI. Please consider upgrading to the latest version.


## 3. Configure Renderer
Set renderer for visualization: `"vscode"`, `"colab"`, or `"browser"`.

In [3]:
renderer = "vscode"

## 4. Load Building Model
Import room geometry from OBJ file.

In [4]:
# Import the Gleis21 standard-floor room geometry from OBJ file
objects_rooms = Topology.ByOBJPath(r"Assignment_04_Giovanni-Carlo-Volpe_rooms.obj")

print(objects_rooms)
print("Building model loaded successfully")
print(f"  - Rooms: {len(objects_rooms)}")

[<topologic_core.Cluster object at 0x00000177098C4730>, <topologic_core.Cluster object at 0x000001770905B0B0>, <topologic_core.Cluster object at 0x0000017768E74830>, <topologic_core.Cluster object at 0x00000177693F6830>, <topologic_core.Cluster object at 0x000001777FB959F0>, <topologic_core.Cluster object at 0x0000017709295FB0>, <topologic_core.Cluster object at 0x000001774A7DD230>, <topologic_core.Cluster object at 0x000001774A7D53F0>, <topologic_core.Cluster object at 0x000001770932E5B0>, <topologic_core.Cluster object at 0x000001770932E6B0>, <topologic_core.Cluster object at 0x0000017768E6D230>, <topologic_core.Cluster object at 0x000001774A784D70>, <topologic_core.Cluster object at 0x0000017769384270>, <topologic_core.Cluster object at 0x0000017748F569B0>, <topologic_core.Cluster object at 0x00000177689C9030>, <topologic_core.Cluster object at 0x0000017768E6FB70>, <topologic_core.Cluster object at 0x000001777F9CA770>, <topologic_core.Cluster object at 0x000001770A8C5770>, <topologi

## 5. Classify Rooms and Assign Attributes
Categorize rooms by type (balcony, bath, bedroom, corridor, elevator, kitchen, livingroom,
stairs, storeroom, wc) and assign a distinct color per type for visualization.

In [ ]:
cells = []
selectors = []

# Color map: one distinct color per Gleis21 room type (per the legend).
# Room names follow the convention "type_NN" (e.g. "wc_03", "balcony_00"),
# so we strip the trailing "_NN" index to recover the room type.
room_colors = {
    "balcony": "lightgreen",   "bath": "paleturquoise", "bedroom": "green",
    "corridor": "salmon",      "elevator": "orangered", "kitchen": "lightsteelblue",
    "livingroom": "magenta",   "stairs": "yellow",      "storeroom": "orange",
    "wc": "blue",
}

for object in objects_rooms:
    d = Topology.Dictionary(object)
    faces = Topology.Faces(object)
    
    # Process only objects with more than one face (volumetric cells)
    if len(faces) > 1:
        c = Cell.ByFaces(faces)
        c = Topology.RemoveCollinearEdges(c)
        s = Topology.InternalVertex(c)
        
        # Extract the name property from the topology dictionary
        name = Dictionary.ValueAtKey(d, "name")     # e.g. "wc_03"
        
        # Assign a color based on the room type (prefix before the "_NN" index)
        room_type = name.rsplit("_", 1)[0]          # e.g. "wc"
        color = room_colors.get(room_type, "grey")  # grey = unexpected type
            
        # Update dictionary with visualization attributes
        d = Dictionary.SetValuesAtKeys(d, ["color", "vertex_size"], [color, 20])
        s = Topology.SetDictionary(s, d)
        
        # Store results for downstream analysis or visualization
        selectors.append(s)
        cells.append(c)
        print(Dictionary.Keys(d), Dictionary.Values(d))

# Output the total count of processed cells
print(len(cells))

## 6. Create Building Complex
Assemble the rooms into a unified cell complex and transfer metadata.

In [ ]:
# Group the processed room cells into a single cluster (Gleis21 Standard Floor).
# NOTE: the rooms are individual closed boxes that only touch (they do not share
# wall faces), so CellComplex.ByCells cannot weld them into a manifold complex and
# returns None. A Cluster preserves every room cell and works downstream with
# Graph.ByTopology and AddApertures. Room-to-room connectivity is therefore derived
# from the door/threshold apertures (see steps 10-11), which is the architecturally
# correct model.
gleis21SF = Cluster.ByTopologies(cells)

# Transfer dictionaries from selectors to the corresponding cells in the cluster
gleis21SF = Topology.TransferDictionariesBySelectors(gleis21SF, selectors, tranCells=True)

# Retrieve all room cells within the cluster
gleis21SF_cells = Topology.Cells(gleis21SF)

# Iterate through each cell to verify the transferred data
for gleis21SF_cell in gleis21SF_cells:
    d = Topology.Dictionary(gleis21SF_cell)
    print(Dictionary.Keys(d), Dictionary.Values(d))

## 7. Visualize Building Complex
Display the floor geometry with color-coded rooms.

In [ ]:
Topology.Show(gleis21SF_cells, faceColorKey="color", faceOpacity=0.9, opacityKey="nothing", faceOpacityKey="nothing", backgroundColor="white")

## 8. Generate Initial Graph
Create one graph node per room from the building topology. Because the rooms are separate
boxes that do not share wall faces, these nodes start out unconnected — the door-based
connectivity between rooms is added in steps 10–11.

In [ ]:
g = Graph.ByTopology(gleis21SF)
verts = Graph.Vertices(g)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

## 9. Display Graph
Visualize the graph with vertex attributes.

In [ ]:
Topology.Show(g, gleis21SF, vertexSizeKey="vertex_size", vertexColorKey="color", backgroundColor="white")

## 10. Load Door & Threshold Apertures
Import doors and thresholds and add them as apertures to the model. Both serve the same
function (openings connecting spaces), so both are added as apertures; they are only
distinguished visually — doors are drawn orange, thresholds blue.

In [ ]:
def load_apertures(path, aperture_type, color):
    """Load each group of an OBJ as a single-face aperture, tagged by type and color."""
    result = []
    for object in Topology.ByOBJPath(path):
        fs = Topology.Faces(object)
        if not fs:                 # guard: skip any group with no face
            continue
        face = Topology.RemoveCollinearEdges(fs[0])
        d = Dictionary.ByKeysValues(["type", "color", "vertex_size"],
                                    [aperture_type, color, 20])
        face = Topology.SetDictionary(face, d)
        result.append(face)
    return result

# Doors and thresholds are both apertures (openings); only their display color differs.
doors      = load_apertures(r"Assignment_04_Giovanni-Carlo-Volpe_doors.obj",      "door",      "orange")
thresholds = load_apertures(r"Assignment_04_Giovanni-Carlo-Volpe_thresholds.obj", "threshold", "blue")
apertures  = doors + thresholds

print(f"Doors: {len(doors)}, Thresholds: {len(thresholds)}, Total apertures: {len(apertures)}")

In [ ]:
house = Topology.AddApertures(gleis21SF, apertures, subTopologyType="face")


## 11. Generate Graph with Apertures
Create graph with shared apertures and exterior connections.

In [ ]:
g = Graph.ByTopology(gleis21SF, direct=False, viaSharedApertures=True, toExteriorApertures=True)
verts = Graph.Vertices(g)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

## 12. Final Visualization
Display complete building model with graph, rooms, and apertures.

In [ ]:
Topology.Show(gleis21SF, apertures, g, vertexSizeKey="vertex_size", vertexColorKey="color", backgroundColor="white")